<a href="https://colab.research.google.com/github/desouki76/Ahmed/blob/main/Copy_of_Untitled3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Bedrock_client_V3**

In [ ]:
"""Amazon Bedrock client for Claude (Jarvis planner).

P2-R1: Native Converse API implementation.

- ask_bedrock(prompt) -> str          : signature preserved, now uses converse().
- ask_bedrock_structured(...) -> dict : NEW. Forced tool-use; JSON arrives as a
                                        structured dict. No regex, no parsing.
- extract_planner_json_object(...)    : kept temporarily so unmigrated call
                                        sites keep working; removed at P2-R1.7.

Retry: exponential backoff on ThrottlingException (3 attempts: 1s, 2s, 4s).
Token usage from every converse() response is logged, ready for P2-R4 cost
tracking to consume.
"""

from __future__ import annotations

import json
import logging
import os
import re
import time
from typing import Any

logger = logging.getLogger(__name__)

DEFAULT_REGION = "us-east-1"
DEFAULT_MODEL_ID = "anthropic.claude-3-sonnet-20240229-v1:0"

_MAX_ATTEMPTS = 3
_BASE_DELAY_S = 1.0
_RETRYABLE_CODES = {"ThrottlingException", "ServiceUnavailableException", "ModelTimeoutException"}


def _bedrock_region() -> str:
    return (os.environ.get("JARVIS_BEDROCK_REGION") or DEFAULT_REGION).strip()


def _model_id() -> str:
    return (os.environ.get("JARVIS_BEDROCK_MODEL_ID") or DEFAULT_MODEL_ID).strip()


def _client():
    import boto3  # noqa: PLC0415 — optional failure surface for tests without AWS

    return boto3.client("bedrock-runtime", region_name=_bedrock_region())


def _converse_with_retry(client, **kwargs: Any) -> dict[str, Any] | None:
    """Call converse() with exponential backoff on retryable errors."""
    from botocore.exceptions import BotoCoreError, ClientError

    for attempt in range(1, _MAX_ATTEMPTS + 1):
        try:
            return client.converse(**kwargs)
        except ClientError as e:
            code = e.response.get("Error", {}).get("Code", "")
            if code in _RETRYABLE_CODES and attempt < _MAX_ATTEMPTS:
                delay = _BASE_DELAY_S * (2 ** (attempt - 1))
                logger.warning(
                    "Bedrock converse retryable error %s (attempt %d/%d), retrying in %.1fs",
                    code, attempt, _MAX_ATTEMPTS, delay,
                )
                time.sleep(delay)
                continue
            logger.warning("Bedrock converse failed: %s", e)
            return None
        except (BotoCoreError, OSError) as e:
            logger.warning("Bedrock converse failed: %s", e)
            return None
    return None


def _log_usage(response: dict[str, Any], model_id: str) -> None:
    """Log token usage — the data feed for P2-R4 cost tracking."""
    usage = response.get("usage") or {}
    logger.info(
        "bedrock_usage model=%s input_tokens=%s output_tokens=%s total_tokens=%s",
        model_id,
        usage.get("inputTokens"),
        usage.get("outputTokens"),
        usage.get("totalTokens"),
    )


def ask_bedrock(prompt: str, model_id: str | None = None) -> str:
    """
    Send a prompt to Claude on Bedrock and return assistant text.

    Signature-compatible with the original invoke_model implementation.
    On failure (credentials, API, network), logs and returns an empty string.
    """
    text = (prompt or "").strip()
    if not text:
        logger.warning("ask_bedrock called with empty prompt")
        return ""

    try:
        client = _client()
    except ImportError as e:
        logger.warning("boto3 not available: %s", e)
        return ""

    mid = (model_id or _model_id()).strip()
    response = _converse_with_retry(
        client,
        modelId=mid,
        messages=[{"role": "user", "content": [{"text": text}]}],
        inferenceConfig={"maxTokens": 4096},
    )
    if response is None:
        return ""

    _log_usage(response, mid)

    content = (((response.get("output") or {}).get("message") or {}).get("content")) or []
    parts = [str(b.get("text") or "") for b in content if isinstance(b, dict) and "text" in b]
    result = "".join(parts).strip()
    if not result:
        logger.warning("Bedrock converse response had no assistant text")
    return result


def ask_bedrock_structured(
    prompt: str,
    schema: dict[str, Any],
    tool_name: str = "emit_result",
    tool_description: str = "Return the structured result.",
    model_id: str | None = None,
) -> dict[str, Any] | None:
    """
    P2-R1 core: get structured JSON via forced tool-use. No text parsing.

    `schema` is a JSON Schema object describing the expected result. Bedrock
    forces the model to call the tool, and the arguments arrive as a Python
    dict. Returns None on any failure.
    """
    text = (prompt or "").strip()
    if not text:
        logger.warning("ask_bedrock_structured called with empty prompt")
        return None

    try:
        client = _client()
    except ImportError as e:
        logger.warning("boto3 not available: %s", e)
        return None

    mid = (model_id or _model_id()).strip()
    response = _converse_with_retry(
        client,
        modelId=mid,
        messages=[{"role": "user", "content": [{"text": text}]}],
        inferenceConfig={"maxTokens": 4096},
        toolConfig={
            "tools": [
                {
                    "toolSpec": {
                        "name": tool_name,
                        "description": tool_description,
                        "inputSchema": {"json": schema},
                    }
                }
            ],
            "toolChoice": {"tool": {"name": tool_name}},
        },
    )
    if response is None:
        return None

    _log_usage(response, mid)

    content = (((response.get("output") or {}).get("message") or {}).get("content")) or []
    for block in content:
        if isinstance(block, dict) and "toolUse" in block:
            tool_input = (block["toolUse"] or {}).get("input")
            if isinstance(tool_input, dict):
                return tool_input
            logger.warning("toolUse input was not a dict: %r", type(tool_input))
            return None

    logger.warning("Bedrock converse response contained no toolUse block")
    return None


# --------------------------------------------------------------------------
# DEPRECATED — kept only until all 7 call sites migrate (removed at P2-R1.7).
# --------------------------------------------------------------------------

def extract_planner_json_object(text: str) -> dict[str, Any] | None:
    """DEPRECATED (P2-R1): use ask_bedrock_structured instead."""
    raw = (text or "").strip()
    if not raw:
        return None
    fence = re.search(r"```(?:json)?\s*([\s\S]*?)```", raw)
    if fence:
        raw = fence.group(1).strip()
    obj = _try_parse_json_object(raw)
    if obj is not None:
        return obj
    balanced = _extract_first_balanced_object(raw)
    if balanced:
        obj = _try_parse_json_object(balanced)
        if obj is not None:
            return obj
    for i, ch in enumerate(raw):
        if ch != "{":
            continue
        for j in range(len(raw), i + 1, -1):
            if j <= i or raw[j - 1] != "}":
                continue
            obj = _try_parse_json_object(raw[i:j])
            if obj is not None:
                return obj
    return None


def _try_parse_json_object(s: str) -> dict[str, Any] | None:
    try:
        val = json.loads(s)
    except json.JSONDecodeError:
        return None
    return val if isinstance(val, dict) else None


def _extract_first_balanced_object(s: str) -> str | None:
    start = -1
    depth = 0
    in_str = False
    escape = False
    for i, c in enumerate(s):
        if in_str:
            if escape:
                escape = False
            elif c == "\\":
                escape = True
            elif c == '"':
                in_str = False
            continue
        if c == '"':
            in_str = True
            continue
        if c == "{":
            if depth == 0:
                start = i
            depth += 1
        elif c == "}":
            depth -= 1
            if depth == 0 and start >= 0:
                return s[start : i + 1]
    return None
